In [3]:
# ============================================================================
# ConvLSTM Model - Riverbank Prediction
# Multi-Sequence Search (4,5,6) → 200 Epochs + Early Stopping
# Strict Temporal Split → Side-by-Side Evaluation
# ============================================================================

# ── Suppress TF / XLA / CUDA warnings BEFORE importing tensorflow ──
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'           # Hide INFO/WARNING/ERROR
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['XLA_FLAGS'] = '--xla_gpu_strict_conv_algorithm_picker=false'
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'

import re
import glob
import sys
import logging
import numpy as np
import pandas as pd

# Silence absl and python logging before TF import
logging.getLogger('tensorflow').setLevel(logging.FATAL)
logging.getLogger('absl').setLevel(logging.FATAL)

import tensorflow as tf
tf.get_logger().setLevel('FATAL')
tf.autograph.set_verbosity(0)

# Suppress remaining warnings
import warnings
warnings.filterwarnings('ignore')
import absl.logging
absl.logging.set_verbosity(absl.logging.FATAL)
absl.logging._warn_preinit_stderr = False

from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
)
from tensorflow.keras.optimizers import Adam
import rasterio
from sklearn.metrics import precision_score, recall_score
from skimage.transform import resize
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

np.random.seed(42)
tf.random.set_seed(42)

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU configured: {len(gpus)} GPU(s) available")
    except RuntimeError as e:
        print(f"GPU Error: {e}")
else:
    print("⚠ No GPU found, using CPU")

# GPU warmup to avoid first-epoch timer warnings
_warmup = tf.zeros((1, 1), dtype=tf.float32)
_ = tf.matmul(_warmup, _warmup)
del _warmup

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_DIR = "/kaggle/input/datasets/kaoserahamed/riverbank/river"
IMG_HEIGHT = 256
IMG_WIDTH = 256
SEQUENCE_LENGTHS = [4, 5, 6]
PREDICTION_HORIZON = 1
STRIDE = 1
EPOCHS = 200
EARLY_STOP_PATIENCE = 20
REDUCE_LR_PATIENCE = 7
BATCH_SIZE = 4
N_COMPONENTS = 3

SETUP_CONFIGS = {
    'Setup 1': {'cutoff_year': 2015, 'test_label': 'Test: 2016-2025'},
    'Setup 2': {'cutoff_year': 2020, 'test_label': 'Test: 2021-2025'},
}

# ============================================================================
# PREPROCESSING FUNCTIONS
# ============================================================================

def keep_largest_n_components_cv2(mask, n=1):
    binary_mask = (mask > 0).astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )
    if num_labels <= 1:
        return np.zeros_like(mask)
    areas = stats[1:, cv2.CC_STAT_AREA]
    sorted_indices = np.argsort(areas)[::-1]
    n_to_keep = min(n, len(sorted_indices))
    top_n_indices = sorted_indices[:n_to_keep]
    cleaned_mask = np.zeros_like(mask)
    for idx in top_n_indices:
        actual_label = idx + 1
        cleaned_mask[labels == actual_label] = mask.max()
    return cleaned_mask


def load_and_preprocess_image(filepath, apply_cleaning=True, n_components=N_COMPONENTS):
    with rasterio.open(filepath) as src:
        img = src.read(1)
    if apply_cleaning:
        img = keep_largest_n_components_cv2(img, n=n_components)
    img_resized = resize(
        img, (IMG_HEIGHT, IMG_WIDTH),
        mode='constant', preserve_range=True, anti_aliasing=False
    )
    img_binary = (img_resized > 0.5).astype(np.float32)
    return img_binary


def create_sequences_with_years(images, years, seq_len):
    X, y, input_years_list, target_years_list = [], [], [], []
    for i in range(0, len(images) - seq_len - PREDICTION_HORIZON + 1, STRIDE):
        X.append(images[i:i + seq_len])
        y.append(images[i + seq_len])
        input_years_list.append(years[i:i + seq_len])
        target_years_list.append(years[i + seq_len])
    return (np.array(X), np.array(y),
            np.array(input_years_list), np.array(target_years_list))


# ============================================================================
# LOSSES AND METRICS
# ============================================================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)


def combined_loss(y_true, y_pred):
    bce = K.mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)


def iou_metric(y_true, y_pred, threshold=0.5):
    y_pred_binary = K.cast(y_pred > threshold, dtype='float32')
    y_true_binary = K.cast(y_true > threshold, dtype='float32')
    intersection = K.sum(y_true_binary * y_pred_binary)
    union = K.sum(y_true_binary) + K.sum(y_pred_binary) - intersection
    return (intersection + K.epsilon()) / (union + K.epsilon())


def dice_coefficient_np(y_true, y_pred, threshold=0.5):
    y_true_b = (y_true > threshold).astype(np.float32)
    y_pred_b = (y_pred > threshold).astype(np.float32)
    intersection = np.sum(y_true_b * y_pred_b)
    return (2. * intersection) / (np.sum(y_true_b) + np.sum(y_pred_b) + 1e-6)


def iou_np(y_true, y_pred, threshold=0.5):
    y_true_b = (y_true > threshold).astype(np.float32)
    y_pred_b = (y_pred > threshold).astype(np.float32)
    intersection = np.sum(y_true_b * y_pred_b)
    union = np.sum(y_true_b) + np.sum(y_pred_b) - intersection
    return intersection / (union + 1e-6)


def calculate_area_difference(y_true, y_pred, pixel_area_km2, threshold=0.5):
    true_area = np.sum((y_true > threshold).astype(np.float32)) * pixel_area_km2
    pred_area = np.sum((y_pred > threshold).astype(np.float32)) * pixel_area_km2
    return pred_area - true_area


# ============================================================================
# CUSTOM STRUCTURED TRAINING CALLBACK (replaces verbose=1)
# ============================================================================

class StructuredTrainingLogger(Callback):
    """
    Compact one-line-per-epoch logger.  Prints a header once, then
    one row per epoch with aligned columns.  Stars (*) mark best val_loss.
    """

    def __init__(self, total_epochs):
        super().__init__()
        self.total_epochs = total_epochs
        self.best_val_loss = np.inf

    def on_train_begin(self, logs=None):
        header = (
            f"{'Ep':>4s}/{'Tot':<4s} │ {'Loss':>8s} │ {'VLoss':>8s} │ "
            f"{'Dice':>6s} │ {'VDice':>6s} │ {'IoU':>6s} │ "
            f"{'VIoU':>6s} │ {'LR':>9s} │ Note"
        )
        print("─" * len(header))
        print(header)
        print("─" * len(header))

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = logs.get('loss', 0)
        vloss = logs.get('val_loss', 0)
        dice = logs.get('dice_coefficient', 0)
        vdice = logs.get('val_dice_coefficient', 0)
        iou = logs.get('iou_metric', 0)
        viou = logs.get('val_iou_metric', 0)
        lr = float(K.get_value(self.model.optimizer.learning_rate))

        note = ""
        if vloss < self.best_val_loss:
            self.best_val_loss = vloss
            note = "★ saved"

        print(
            f"{epoch+1:>4d}/{self.total_epochs:<4d} │ {loss:>8.4f} │ {vloss:>8.4f} │ "
            f"{dice:>6.4f} │ {vdice:>6.4f} │ {iou:>6.4f} │ "
            f"{viou:>6.4f} │ {lr:>9.2e} │ {note}"
        )

    def on_train_end(self, logs=None):
        print("─" * 85)
        print(f"  Training finished  │  Best val_loss: {self.best_val_loss:.4f}")
        print("─" * 85)


# ============================================================================
# CALLBACKS FACTORY
# ============================================================================

def create_callbacks(model_name, checkpoint_dir='checkpoints'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    # Use .keras format to avoid HDF5 legacy warnings
    ckpt_path = os.path.join(checkpoint_dir, f'{model_name}_best.keras')
    return [
        ModelCheckpoint(
            ckpt_path,
            monitor='val_loss', save_best_only=True,
            save_weights_only=False, mode='min', verbose=0   # silent
        ),
        EarlyStopping(
            monitor='val_loss', patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True, verbose=0              # silent
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=REDUCE_LR_PATIENCE, min_lr=1e-7, verbose=0  # silent
        ),
        StructuredTrainingLogger(total_epochs=EPOCHS),
    ]


# ============================================================================
# MODEL DEFINITION (same structure, parametrized by seq_len)
# ============================================================================

def build_convlstm_model(seq_len):
    input_shape = (seq_len, IMG_HEIGHT, IMG_WIDTH, 1)
    inputs = layers.Input(shape=input_shape)

    x = layers.TimeDistributed(
        layers.Conv2D(32, 3, padding='same', activation='relu'))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.MaxPooling2D(2))(x)
    x = layers.TimeDistributed(layers.Dropout(0.2))(x)

    x = layers.TimeDistributed(
        layers.Conv2D(64, 3, padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.MaxPooling2D(2))(x)
    x = layers.TimeDistributed(layers.Dropout(0.2))(x)

    x = layers.ConvLSTM2D(
        filters=128, kernel_size=(3, 3), padding='same',
        return_sequences=True, dropout=0.3,
        kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)

    x = layers.ConvLSTM2D(
        filters=64, kernel_size=(3, 3), padding='same',
        return_sequences=False, dropout=0.3,
        kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)

    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(16, 3, padding='same', activation='relu')(x)
    outputs = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name=f'ConvLSTM_seq{seq_len}')
    model.compile(
        optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
        loss=combined_loss,
        metrics=[dice_coefficient, iou_metric]
    )
    return model


# ============================================================================
# STRICT TEMPORAL SPLIT (guarantees no leakage)
# ============================================================================

def prepare_split(X_all, y_all, target_years_all, input_years_all,
                  cutoff_year, all_years_list):
    max_input_year = np.array([iy.max() for iy in input_years_all])

    train_mask = (target_years_all <= cutoff_year) & (max_input_year <= cutoff_year)
    test_mask = target_years_all > cutoff_year

    X_train = X_all[train_mask]
    y_train = y_all[train_mask]
    ty_train = target_years_all[train_mask]

    X_test = X_all[test_mask]
    y_test = y_all[test_mask]
    ty_test = target_years_all[test_mask]

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError(
            f"Empty split! Train={len(X_train)}, Test={len(X_test)} "
            f"with cutoff={cutoff_year}"
        )

    sort_idx = np.argsort(ty_train)
    X_train, y_train, ty_train = (
        X_train[sort_idx], y_train[sort_idx], ty_train[sort_idx]
    )

    split_idx = int(len(X_train) * 0.85)
    X_tr, y_tr, ty_tr = X_train[:split_idx], y_train[:split_idx], ty_train[:split_idx]
    X_val, y_val, ty_val = X_train[split_idx:], y_train[split_idx:], ty_train[split_idx:]

    if len(X_val) == 0:
        raise ValueError("Empty validation set!")

    train_val_years = set(ty_tr.tolist()) | set(ty_val.tolist())
    test_years = set(ty_test.tolist())
    overlap = train_val_years & test_years
    assert len(overlap) == 0, f"DATA LEAK! Overlapping years: {overlap}"

    X_tr  = np.expand_dims(X_tr,  axis=-1)
    y_tr  = np.expand_dims(y_tr,  axis=-1)
    X_val = np.expand_dims(X_val, axis=-1)
    y_val = np.expand_dims(y_val, axis=-1)
    X_test = np.expand_dims(X_test, axis=-1)
    y_test = np.expand_dims(y_test, axis=-1)

    print(f"  Train : {len(X_tr):>4d} samples  (years {ty_tr.min()}-{ty_tr.max()})")
    print(f"  Val   : {len(X_val):>4d} samples  (years {ty_val.min()}-{ty_val.max()})")
    print(f"  Test  : {len(X_test):>4d} samples  (years {ty_test.min()}-{ty_test.max()})")
    print(f"  ✓ No year overlap between train/val and test sets")

    return X_tr, y_tr, X_val, y_val, X_test, y_test, ty_test


# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_model(model, X_test, y_test, target_years_test,
                   pixel_area_km2, model_name, setup_name):
    pred = model.predict(X_test, verbose=0)
    persistence_pred = X_test[:, -1, :, :, :]

    results = []
    for name, predictions in [('Persistence', persistence_pred),
                               (model_name, pred)]:
        for i in range(len(y_test)):
            yt = y_test[i, :, :, 0]
            yp = predictions[i, :, :, 0]
            year = target_years_test[i]

            iou = iou_np(yt, yp)
            dice = dice_coefficient_np(yt, yp)
            yt_flat = (yt.flatten() > 0.5).astype(int)
            yp_flat = (yp.flatten() > 0.5).astype(int)
            prec = precision_score(yt_flat, yp_flat, zero_division=0)
            rec = recall_score(yt_flat, yp_flat, zero_division=0)
            area_diff = calculate_area_difference(yt, yp, pixel_area_km2)

            results.append({
                'Setup': setup_name, 'Model': name, 'Year': year,
                'IoU': iou, 'Dice': dice, 'Precision': prec,
                'Recall': rec, 'Area_Diff_km2': area_diff
            })

    return pd.DataFrame(results)


# ============================================================================
# DATA LOADING
# ============================================================================

print("=" * 80)
print("LOADING DATA")
print("=" * 80)

files = sorted(glob.glob(os.path.join(DATA_DIR, "*.tif")))
file_info = []
for filepath in files:
    filename = os.path.basename(filepath)
    match = re.search(r'(\d{4})', filename)
    if match:
        file_info.append({
            'filepath': filepath, 'filename': filename,
            'year': int(match.group(1))
        })

df_files = pd.DataFrame(file_info).sort_values('year').reset_index(drop=True)
print(f"Found {len(df_files)} files, years {df_files['year'].min()}-{df_files['year'].max()}")

all_images, all_years = [], []
for idx, row in df_files.iterrows():
    img = load_and_preprocess_image(row['filepath'])
    all_images.append(img)
    all_years.append(row['year'])
    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx+1}/{len(df_files)} images...")

print(f"✓ Loaded {len(all_images)} images")

# Pixel area calculation
with rasterio.open(df_files.iloc[0]['filepath']) as src:
    bounds = src.bounds
    center_lat = (bounds.top + bounds.bottom) / 2
    if src.crs and src.crs.to_epsg() == 4326:
        meters_per_deg_lon = 111320 * np.cos(np.radians(center_lat))
        width_m = (bounds.right - bounds.left) * meters_per_deg_lon
        height_m = (bounds.top - bounds.bottom) * 111320
        total_area_km2 = (width_m * height_m) / 1e6
    else:
        total_area_km2 = (
            (bounds.right - bounds.left) * (bounds.top - bounds.bottom)
        ) / 1e6

pixel_area_km2 = total_area_km2 / (IMG_HEIGHT * IMG_WIDTH)
print(f"✓ Pixel area: {pixel_area_km2:.8f} km²")

# ============================================================================
# MULTI-SEQUENCE SEARCH + BOTH SETUPS
# ============================================================================

all_results = []
summary_rows = []
best_configs = {}
training_histories = {}

for seq_len in SEQUENCE_LENGTHS:
    print("\n" + "█" * 80)
    print(f"  SEQUENCE LENGTH = {seq_len}")
    print("█" * 80)

    X_all_seq, y_all_seq, input_years_all, target_years_all = \
        create_sequences_with_years(all_images, all_years, seq_len)
    print(f"✓ {len(X_all_seq)} sequences  |  target years "
          f"{target_years_all.min()}-{target_years_all.max()}")

    for setup_name, cfg in SETUP_CONFIGS.items():
        cutoff = cfg['cutoff_year']
        label = cfg['test_label']
        tag = f"seq{seq_len}_{setup_name.replace(' ', '')}"

        print(f"\n{'─'*70}")
        print(f"  {setup_name} (cutoff={cutoff})  |  SeqLen={seq_len}")
        print(f"{'─'*70}")

        try:
            X_tr, y_tr, X_val, y_val, X_test, y_test, ty_test = \
                prepare_split(
                    X_all_seq, y_all_seq, target_years_all,
                    input_years_all, cutoff, all_years
                )
        except ValueError as e:
            print(f"  ⚠ Skipping: {e}")
            continue

        # ---- Build & train ----
        K.clear_session()
        tf.random.set_seed(42)
        np.random.seed(42)

        model = build_convlstm_model(seq_len)
        if seq_len == SEQUENCE_LENGTHS[0] and setup_name == list(SETUP_CONFIGS.keys())[0]:
            model.summary()

        print(f"\n  Training {tag}  (max {EPOCHS} epochs, "
              f"early-stop patience={EARLY_STOP_PATIENCE})\n")

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=create_callbacks(tag),
            verbose=0    # ← fully silent; our callback handles output
        )
        training_histories[tag] = history

        stopped_epoch = len(history.history['loss'])
        best_val_loss = min(history.history['val_loss'])
        best_val_dice = max(history.history['val_dice_coefficient'])
        best_val_iou = max(history.history['val_iou_metric'])

        print(f"\n  ✓ Stopped at epoch {stopped_epoch}/{EPOCHS}  |  "
              f"Best val_loss={best_val_loss:.4f}  "
              f"val_dice={best_val_dice:.4f}  val_iou={best_val_iou:.4f}")

        # ---- Evaluate ----
        model_label = f'ConvLSTM(seq={seq_len})'
        df_res = evaluate_model(
            model, X_test, y_test, ty_test,
            pixel_area_km2, model_label,
            f'{setup_name} ({label})'
        )
        df_res['SeqLen'] = seq_len
        all_results.append(df_res)

        for mdl_name in df_res['Model'].unique():
            sub = df_res[df_res['Model'] == mdl_name]
            summary_rows.append({
                'SeqLen': seq_len,
                'Setup': setup_name,
                'Model': mdl_name,
                'Epochs_Run': stopped_epoch,
                'Best_Val_Loss': best_val_loss if mdl_name != 'Persistence' else np.nan,
                'IoU': sub['IoU'].mean(),
                'Dice': sub['Dice'].mean(),
                'Precision': sub['Precision'].mean(),
                'Recall': sub['Recall'].mean(),
                'Area_Diff_km2': sub['Area_Diff_km2'].mean(),
                'Abs_Area_Diff_km2': sub['Area_Diff_km2'].abs().mean(),
            })


# ============================================================================
# AGGREGATE RESULTS
# ============================================================================

df_all = pd.concat(all_results, ignore_index=True)
df_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 100)
print("  SEQUENCE LENGTH COMPARISON  –  Mean Test Metrics (ConvLSTM only)")
print("=" * 100)

convlstm_summary = df_summary[df_summary['Model'] != 'Persistence'].copy()
pivot = convlstm_summary.pivot_table(
    index='SeqLen', columns='Setup',
    values=['IoU', 'Dice', 'Precision', 'Recall', 'Abs_Area_Diff_km2'],
    aggfunc='mean'
).round(4)
print(pivot.to_string())

# ============================================================================
# IDENTIFY BEST SEQUENCE LENGTH PER SETUP
# ============================================================================

print("\n" + "=" * 100)
print("  BEST SEQUENCE LENGTH PER SETUP  (by mean IoU on test)")
print("=" * 100)

for setup_name in SETUP_CONFIGS:
    sub = convlstm_summary[convlstm_summary['Setup'] == setup_name]
    if sub.empty:
        continue
    best_row = sub.loc[sub['IoU'].idxmax()]
    best_configs[setup_name] = int(best_row['SeqLen'])
    print(f"  {setup_name}: Best SeqLen = {int(best_row['SeqLen'])}  "
          f"(IoU={best_row['IoU']:.4f}, Dice={best_row['Dice']:.4f})")

# ============================================================================
# SIDE-BY-SIDE: ACTUAL vs PREDICTED (per setup, best seq_len)
# ============================================================================

print("\n" + "=" * 100)
print("  SIDE-BY-SIDE  –  Actual vs Predicted  (Best Config per Setup)")
print("=" * 100)

for setup_name, cfg in SETUP_CONFIGS.items():
    best_seq = best_configs.get(setup_name)
    if best_seq is None:
        continue

    label = cfg['test_label']
    sub = df_all[
        (df_all['Setup'] == f'{setup_name} ({label})') &
        (df_all['SeqLen'] == best_seq)
    ].copy()

    if sub.empty:
        continue

    print(f"\n{'━'*90}")
    print(f"  {setup_name} | Best Sequence Length = {best_seq} | {label}")
    print(f"{'━'*90}")

    years = sorted(sub['Year'].unique())
    models_in_sub = sub['Model'].unique()

    rows = []
    for year in years:
        row = {'Year': year}
        for mdl in models_in_sub:
            m = sub[(sub['Year'] == year) & (sub['Model'] == mdl)]
            if m.empty:
                continue
            m = m.iloc[0]
            prefix = 'Pred' if 'ConvLSTM' in mdl else 'Pers'
            row[f'{prefix}_IoU'] = round(m['IoU'], 4)
            row[f'{prefix}_Dice'] = round(m['Dice'], 4)
            row[f'{prefix}_Precision'] = round(m['Precision'], 4)
            row[f'{prefix}_Recall'] = round(m['Recall'], 4)
            row[f'{prefix}_ΔArea_km²'] = round(m['Area_Diff_km2'], 6)
        rows.append(row)

    df_side = pd.DataFrame(rows)

    desired_order = ['Year']
    for prefix in ['Pred', 'Pers']:
        for metric in ['IoU', 'Dice', 'Precision', 'Recall', 'ΔArea_km²']:
            col = f'{prefix}_{metric}'
            if col in df_side.columns:
                desired_order.append(col)
    df_side = df_side[[c for c in desired_order if c in df_side.columns]]

    print(df_side.to_string(index=False))

    numeric_cols = [c for c in df_side.columns if c != 'Year']
    means = df_side[numeric_cols].mean()
    print(f"{'─'*90}")
    mean_str = "  MEAN |"
    for c in numeric_cols:
        mean_str += f"  {c}={means[c]:.4f}"
    print(mean_str)

# ============================================================================
# FULL SUMMARY TABLE
# ============================================================================

print("\n" + "=" * 100)
print("  FULL SUMMARY TABLE")
print("=" * 100)

display_cols = [
    'SeqLen', 'Setup', 'Model', 'Epochs_Run',
    'IoU', 'Dice', 'Precision', 'Recall',
    'Area_Diff_km2', 'Abs_Area_Diff_km2'
]
print(df_summary[display_cols].round(4).to_string(index=False))

# ============================================================================
# VISUALIZATION: TRAINING CURVES (best seq per setup)
# ============================================================================

fig, axes = plt.subplots(len(SETUP_CONFIGS), 3, figsize=(18, 5 * len(SETUP_CONFIGS)))
if len(SETUP_CONFIGS) == 1:
    axes = axes[np.newaxis, :]

for row_idx, (setup_name, cfg) in enumerate(SETUP_CONFIGS.items()):
    best_seq = best_configs.get(setup_name)
    if best_seq is None:
        continue
    tag = f"seq{best_seq}_{setup_name.replace(' ', '')}"
    hist = training_histories.get(tag)
    if hist is None:
        continue

    ax = axes[row_idx, 0]
    ax.plot(hist.history['loss'], label='Train Loss')
    ax.plot(hist.history['val_loss'], label='Val Loss')
    ax.set_title(f'{setup_name} | seq={best_seq} | Loss')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True)

    ax = axes[row_idx, 1]
    ax.plot(hist.history['dice_coefficient'], label='Train Dice')
    ax.plot(hist.history['val_dice_coefficient'], label='Val Dice')
    ax.set_title(f'{setup_name} | seq={best_seq} | Dice')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Dice'); ax.legend(); ax.grid(True)

    ax = axes[row_idx, 2]
    ax.plot(hist.history['iou_metric'], label='Train IoU')
    ax.plot(hist.history['val_iou_metric'], label='Val IoU')
    ax.set_title(f'{setup_name} | seq={best_seq} | IoU')
    ax.set_xlabel('Epoch'); ax.set_ylabel('IoU'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig('training_curves_best_configs.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Training curves saved → training_curves_best_configs.png")

# ============================================================================
# VISUALIZATION: BAR CHART – Sequence Length Comparison
# ============================================================================

fig, axes = plt.subplots(1, len(SETUP_CONFIGS), figsize=(8 * len(SETUP_CONFIGS), 6))
if len(SETUP_CONFIGS) == 1:
    axes = [axes]

metrics_to_plot = ['IoU', 'Dice', 'Precision', 'Recall']

for ax_idx, (setup_name, cfg) in enumerate(SETUP_CONFIGS.items()):
    ax = axes[ax_idx]
    sub = convlstm_summary[convlstm_summary['Setup'] == setup_name]
    if sub.empty:
        continue

    x = np.arange(len(metrics_to_plot))
    width = 0.2
    for i, sl in enumerate(SEQUENCE_LENGTHS):
        vals = sub[sub['SeqLen'] == sl][metrics_to_plot].values
        if len(vals) == 0:
            continue
        vals = vals[0]
        offset = (i - len(SEQUENCE_LENGTHS) / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=f'seq={sl}')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8)

    best = best_configs.get(setup_name)
    ax.set_title(f'{setup_name} – Sequence Comparison (best={best})')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_to_plot)
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('sequence_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Sequence comparison chart saved → sequence_comparison_bar.png")

# ============================================================================
# VISUALIZATION: SAMPLE PREDICTIONS (best config per setup)
# ============================================================================

for setup_name, cfg in SETUP_CONFIGS.items():
    best_seq = best_configs.get(setup_name)
    if best_seq is None:
        continue

    cutoff = cfg['cutoff_year']
    label = cfg['test_label']
    tag = f"seq{best_seq}_{setup_name.replace(' ', '')}"

    X_all_v, y_all_v, iy_all_v, ty_all_v = \
        create_sequences_with_years(all_images, all_years, best_seq)

    test_mask = ty_all_v > cutoff
    X_test_v = np.expand_dims(X_all_v[test_mask], axis=-1)
    y_test_v = y_all_v[test_mask]
    ty_test_v = ty_all_v[test_mask]

    ckpt = f'checkpoints/{tag}_best.keras'
    if os.path.exists(ckpt):
        best_model = tf.keras.models.load_model(
            ckpt,
            custom_objects={
                'combined_loss': combined_loss,
                'dice_coefficient': dice_coefficient,
                'iou_metric': iou_metric
            }
        )
    else:
        continue

    preds = best_model.predict(X_test_v, verbose=0)
    n_show = min(6, len(y_test_v))
    indices = np.linspace(0, len(y_test_v) - 1, n_show, dtype=int)

    fig, axes_v = plt.subplots(n_show, 3, figsize=(15, 4 * n_show))
    if n_show == 1:
        axes_v = axes_v[np.newaxis, :]

    for row, idx in enumerate(indices):
        actual = y_test_v[idx]
        predicted = (preds[idx, :, :, 0] > 0.5).astype(np.float32)
        diff = np.zeros((*actual.shape, 3))
        diff[:, :, 1] = actual * (1 - predicted)
        diff[:, :, 0] = predicted * (1 - actual)
        diff[:, :, 2] = actual * predicted

        year = ty_test_v[idx]
        iou_val = iou_np(actual, predicted)
        dice_val = dice_coefficient_np(actual, predicted)

        axes_v[row, 0].imshow(actual, cmap='gray')
        axes_v[row, 0].set_title(f'Actual – {year}', fontsize=11)
        axes_v[row, 0].axis('off')

        axes_v[row, 1].imshow(predicted, cmap='gray')
        axes_v[row, 1].set_title(
            f'Predicted – {year}\nIoU={iou_val:.4f}  Dice={dice_val:.4f}',
            fontsize=11
        )
        axes_v[row, 1].axis('off')

        axes_v[row, 2].imshow(diff)
        axes_v[row, 2].set_title(
            f'Difference – {year}\n(R=FP, G=FN, B=TP)', fontsize=11
        )
        axes_v[row, 2].axis('off')

    fig.suptitle(
        f'{setup_name} | seq={best_seq} | Actual vs Predicted',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    fname = f'predictions_{setup_name.replace(" ", "_")}_seq{best_seq}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Prediction samples saved → {fname}")

print("\n" + "=" * 100)
print("  ✓ ALL DONE – ConvLSTM multi-sequence evaluation complete")
print("=" * 100)

✓ GPU configured: 2 GPU(s) available
LOADING DATA
Found 39 files, years 1987-2025
  Processed 10/39 images...
  Processed 20/39 images...
  Processed 30/39 images...
✓ Loaded 39 images
✓ Pixel area: 0.32843740 km²

████████████████████████████████████████████████████████████████████████████████
  SEQUENCE LENGTH = 4
████████████████████████████████████████████████████████████████████████████████
✓ 35 sequences  |  target years 1991-2025

──────────────────────────────────────────────────────────────────────
  Setup 1 (cutoff=2015)  |  SeqLen=4
──────────────────────────────────────────────────────────────────────
  Train :   21 samples  (years 1991-2011)
  Val   :    4 samples  (years 2012-2015)
  Test  :   10 samples  (years 2016-2025)
  ✓ No year overlap between train/val and test sets


Model: "ConvLSTM_seq4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 4, 256, 256, 1) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 4, 256, 256,    │           320 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 4, 256, 256,    │           128 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 4, 128, 128,    │             0 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 4, 128, 128,    │             0 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 4, 128, 128,    │        18,496 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, 4, 128, 128,    │           256 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 4, 64, 64, 64)  │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_7              │ (None, 4, 64, 64, 64)  │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d (ConvLSTM2D)        │ (None, 4, 64, 64, 128) │       885,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 4, 64, 64, 128) │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_1 (ConvLSTM2D)      │ (None, 64, 64, 64)     │       442,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 256, 256, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 256, 256, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 1,408,257 (5.37 MB)

 Trainable params: 1,407,489 (5.37 MB)

 Non-trainable params: 768 (3.00 KB)


  Training seq4_Setup1  (max 200 epochs, early-stop patience=20)

──────────────────────────────────────────────────────────────────────────────────────
  Ep/Tot  │     Loss │    VLoss │   Dice │  VDice │    IoU │   VIoU │        LR │ Note
──────────────────────────────────────────────────────────────────────────────────────


2026-04-18 03:53:34.181010: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-18 03:53:34.329506: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-18 03:53:34.733720: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-18 03:53:34.899209: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-18 03:53:35.177770: E external/local_xla/xla/stream_

   1/200  │   2.3070 │   1.9220 │ 0.0612 │ 0.0511 │ 0.0278 │ 0.2049 │  1.00e-04 │ ★ saved
   2/200  │   2.0863 │   1.9045 │ 0.0654 │ 0.0520 │ 0.0321 │ 0.2891 │  1.00e-04 │ ★ saved
   3/200  │   1.8986 │   1.8790 │ 0.0706 │ 0.0535 │ 0.0510 │ 0.3127 │  1.00e-04 │ ★ saved
   4/200  │   1.7585 │   1.8489 │ 0.0788 │ 0.0551 │ 0.1162 │ 0.3318 │  1.00e-04 │ ★ saved
   5/200  │   1.6671 │   1.8130 │ 0.0933 │ 0.0569 │ 0.2178 │ 0.3385 │  1.00e-04 │ ★ saved
   6/200  │   1.6035 │   1.7725 │ 0.1077 │ 0.0588 │ 0.2838 │ 0.3044 │  1.00e-04 │ ★ saved
   7/200  │   1.5530 │   1.7314 │ 0.1160 │ 0.0595 │ 0.3204 │ 0.1929 │  1.00e-04 │ ★ saved
   8/200  │   1.5088 │   1.6951 │ 0.1233 │ 0.0595 │ 0.3491 │ 0.1347 │  1.00e-04 │ ★ saved
   9/200  │   1.4663 │   1.6562 │ 0.1356 │ 0.0601 │ 0.3768 │ 0.1140 │  1.00e-04 │ ★ saved
  10/200  │   1.4272 │   1.6184 │ 0.1476 │ 0.0595 │ 0.3955 │ 0.0772 │  1.00e-04 │ ★ saved
  11/200  │   1.3899 │   1.5813 │ 0.1574 │ 0.0591 │ 0.4098 │ 0.0504 │  1.00e-04 │ ★ saved
  12/200  